# Exercise 2: From the Pair Classifier to FPGA Firmware with _hls4ml_

## Objectives
In this exercise, you will:
1. Rebuild the pair classifier in PyTorch, with its trained weights
2. Choose the fixed-point precision, from one type for the whole model to one type for every layer, with the profiling tools of _hls4ml_, and check it in C simulation
3. Estimate the FPGA resources and the latency with Vitis HLS, and see the effect of the reuse factor
4. See how the provided custom kernel is connected to the _hls4ml_ network
5. Get the prebuilt bitstream, and compare the real resources with the estimates

## Instructions
Complete the code cells marked with `# TODO` comments. Follow the hints provided.

## Part 1: Environment Setup
Run the following cells to set up the environment (no changes needed). <br>
This exercise needs the Xilinx tools (Vitis HLS for the synthesis) and XRT (`xclbinutil`). It uses the files written by Exercise 1 in `work/`.

In [ ]:
# Load libraries and import packages (no changes needed)

# General imports
import sys
import re
import shutil
import subprocess
from pathlib import Path

# Numpy and plotting
import numpy as np
import matplotlib.pyplot as plt

# PyTorch and hls4ml
import torch
import torch.nn as nn
import hls4ml
from hls4ml.model.profiling import numerical    # the profiling tool of hls4ml (Part 4)

# The folder of the course: the one with utils.py and hi_common.py, above the ex/ and sol/ folders
ROOT_DIR = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "utils.py").exists() and (folder / "hi_common.py").exists():
        ROOT_DIR = folder
        break
if ROOT_DIR is None:
    raise FileNotFoundError("utils.py and hi_common.py not found: open this notebook from the ex/ or sol/ folder "
                            "of the course")
sys.path.insert(0, str(ROOT_DIR))

# Local file with some useful methods (plots, helpers)
from utils import *

# Configuration and folders
CONFIG = course_config(ROOT_DIR)                # the settings of the course (in utils.py)
WORK_DIR = ROOT_DIR / "work"
DATA_DIR = WORK_DIR / "data"          # written by Exercise 1
HLS_DIR = WORK_DIR / "hls"            # hls4ml projects
FPGA_DIR = WORK_DIR / "fpga"          # the prebuilt bitstream
FIGURE_DIR = WORK_DIR / "figures"
for folder in (HLS_DIR, FPGA_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

FPGA_PART = CONFIG["hls"]["part"]     # FPGA part of the Alveo U55C        - DO NOT CHANGE
CLOCK_PERIOD_NS = 1000.0 / 300        # 300 MHz clock                      - DO NOT CHANGE
THRESHOLD = CONFIG["operating_threshold"]

use_course_style()
tools_found = {tool: shutil.which(tool) is not None for tool in ("vitis_hls", "g++", "xclbinutil")}
print(f"hls4ml {hls4ml.__version__}, torch {torch.__version__}")
print(f"Tools found: {tools_found}")
print("Environment setup correctly!" if all(tools_found.values()) else "Some tools are missing: vitis_hls, g++ and xclbinutil must be in the PATH")

In Exercise 1 you split the trained model, and saved the weights of the pair classifier and some real pair features in `work/data/`. Now the pair classifier goes to the FPGA.

## Part 2: The Pair Classifier in PyTorch
_hls4ml_ reads the model through its PyTorch frontend. The pair classifier is the small MLP of Exercise 0: 64 inputs, 64 hidden neurons with a ReLU, 2 outputs.

In [ ]:
# Define the pair classifier and load the reference pairs (no changes needed)
class PairClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 2)

    def forward(self, x):          # hls4ml names the input layer after this argument: "x"
        return self.fc2(self.relu(self.fc1(x)))


calibration_pairs = np.load(DATA_DIR / "pairs_validation.npz")   # to measure the ranges of the values
test_pairs = np.load(DATA_DIR / "pairs_test.npz")                # to measure the accuracy
calibration_features = calibration_pairs["X"]
test_features = np.ascontiguousarray(test_pairs["X"])
float_logits = test_pairs["logits"]                              # pair_head.onnx on the test pairs
print(f"Calibration pairs: {calibration_features.shape}, test pairs: {test_features.shape}")

### Task 2.1: Load the Trained Weights

In [ ]:
# TODO: load the trained weights of the pair classifier into the PyTorch model

# Hint: np.load(DATA_DIR / 'pair_head_weights.npz') gives the arrays W1, b1, W2, b2 saved by Exercise 1
# Hint: ONNX stores a weight matrix as [inputs, outputs], PyTorch's nn.Linear as [outputs, inputs]:
#       transpose it with .T (and .copy() it, to make it contiguous)
# Hint: inside torch.no_grad(), model.fc1.weight.copy_(torch.from_numpy(array)) copies an array into a parameter

weights = None  # Replace with correct code
model = PairClassifier().eval()
with torch.no_grad():
    # YOUR CODE HERE
    pass

with torch.no_grad():
    torch_logits = model(torch.from_numpy(test_features)).numpy()
print(model)
print(f"PyTorch vs pair_head.onnx on {len(test_features):,} test pairs: largest difference in the logits "
      f"{np.abs(torch_logits - float_logits).max():.1e}")

## Part 3: Fixed-Point Precision

### Preamble: Numerical Precision, and Why the FPGA Works with Integers
**What numerical precision is.** A computer stores every number in a fixed number of bits, so it can only represent a finite set of values: any other value is rounded to the nearest one that it can represent. The *precision* of a number format is how many bits it uses, and how it spends them between two things:
- the **range**: how large a value can be before it no longer fits (an *overflow*)
- the **resolution**: how close two neighbouring values are, and so how large the rounding error can be

CPUs and GPUs use **floating point**, usually `float32`: 1 sign bit, 8 bits of exponent and 23 bits of mantissa, for x = ± mantissa × 2<sup>exponent</sup>. The exponent moves the binary point to follow the value, so the format covers from about 10<sup>−38</sup> to 10<sup>38</sup>, always with about 7 significant digits. It is very convenient, and the hardware for it is already in the chip.

**Why integers on an FPGA.** On an FPGA nothing is already there: every operation of the network becomes a piece of circuit, and you pay for it in resources.
- **Float arithmetic is expensive.** To add two floats, the circuit must compare the exponents, shift one of the mantissas, add, normalise the result and round it. A single `float32` multiplier or adder costs a few DSP slices or hundreds of LUTs, and takes several clock cycles. An integer adder is a simple chain of logic, and an integer multiplication of up to 27 × 18 bits fits in **one** DSP slice.
- **Every bit costs.** On an FPGA you choose the width of every number. An adder grows with the number of bits, and a multiplier built from LUTs with the product of the widths of its two inputs: halving the bits makes it about four times smaller. With 4,224 multiplications per pair, this decides whether the design fits on the chip.
- **Integer arithmetic is exact.** Additions and multiplications of integers do not round, so the result does not depend on the order of the operations. This is why a C simulation on the CPU and the FPGA give the same results **bit for bit**, and so do two different circuits that do the same integer operations: you will see it in Part 6.
- **Neural networks do not need `float32`.** The trained weights and the activations do not need 7 significant digits: 8 to 16 bits are usually enough to take the same decisions as the float model. How few bits are enough is exactly what this part is about.

**Fixed point: integers with an implicit scale.** A fixed-point number is an integer *q* with an agreed scale: the value it stands for is x = q × 2<sup>−F</sup>, where *F* is the number of **fractional bits**. Unlike in a float, the scale is not stored in the number: it is fixed (hence the name) when the design is made, once per tensor, and it lives only in the type `ap_fixed<W, I>`, with *W* bits in total, *I* integer bits (sign included) and *F* = *W* − *I*. The FPGA only ever sees integers.

### A Concrete Example: `ap_fixed<8,3>`
*W* = 8 bits in total, *I* = 3 integer bits (sign included), so *F* = 5 fractional bits, and one step is 2<sup>−5</sup> = 0.03125: the type covers the values from −4 to 3.96875.

![The number 1.7 in float32 and in ap_fixed<8,3>, and the values that ap_fixed<8,3> can hold](../images/fixed_point.png)

**From float to integer:** multiply by 2<sup>F</sup> and round to the nearest integer. For x = 1.7:

&emsp; 1.7 × 2<sup>5</sup> = 54.4 &ensp;→&ensp; q = **54** &ensp;→&ensp; bits `001.10110` (the binary point is only in our heads, the FPGA stores `00110110`)

**From integer to float:** divide by 2<sup>F</sup>.

&emsp; 54 / 2<sup>5</sup> = **1.6875**, an error of 1.7 − 1.6875 = 0.0125: never more than half a step (0.015625)

**A multiplication** is an integer multiplication followed by a shift. Take a weight w = 0.625, that is q<sub>w</sub> = 0.625 × 2<sup>5</sup> = 20:

&emsp; 20 × 54 = 1080, a number with 2F = 10 fractional bits: 1080 / 2<sup>10</sup> = 1.0546875, exactly 0.625 × 1.6875

To go back to *F* = 5, drop the 5 extra bits (a shift by 5) and round: 1080 / 2<sup>5</sup> = 33.75 → 34 → 34 / 2<sup>5</sup> = 1.0625. The FPGA only did an integer multiplication and a shift.

**Out of range:** for x = 5.2, 5.2 × 2<sup>5</sup> = 166.4 → 166, which does not fit in 8 bits (at most 127). With `AP_SAT` the value **saturates** to 127, that is 3.96875. With `AP_WRAP`, the default of _hls4ml_, it **wraps around** to 166 − 256 = −90, that is −2.8125: even the sign is wrong! This is why the integer bits must cover the largest value of every tensor.

The cell below does the same round trip in Python, for a few values and two types: `ap_fixed<8,3>` and `ap_fixed<16,6>`, the default type of _hls4ml_ that you met in Exercise 0.

In [ ]:
# From float to integer and back, for two fixed-point types (no changes needed)
def float_to_fixed(value, W, I):
    """Float -> integer q: multiply by 2^F, round to the nearest integer, and saturate to the W-bit range."""
    F = W - I
    q = int(np.round(value * 2**F))                          # np.round: ties to even, as AP_RND_CONV
    return min(max(q, -2**(W - 1)), 2**(W - 1) - 1)          # saturate, as AP_SAT


def fixed_to_float(q, W, I):
    """Integer q -> float: divide by 2^F."""
    return q / 2**(W - I)


def bits_of(q, W, I):
    """The W bits that the FPGA stores (two's complement), with the binary point where the type puts it."""
    pattern = format(q & (2**W - 1), f"0{W}b")
    return pattern[:I] + "." + pattern[I:]


def show_round_trip(values, W, I):
    F = W - I
    print(f"ap_fixed<{W},{I}>: F = {F} fractional bits, step 2^-{F} = {2**-F:g}, "
          f"range {fixed_to_float(-2**(W - 1), W, I):g} to {fixed_to_float(2**(W - 1) - 1, W, I):g}")
    print(f"  {'float x':>8} {'x * 2^F':>10} {'integer q':>10} {'bits':>18} {'q / 2^F':>12} {'error':>10}")
    for value in values:
        q = float_to_fixed(value, W, I)
        back = fixed_to_float(q, W, I)
        print(f"  {value:>8} {value * 2**F:>10.2f} {q:>10} {bits_of(q, W, I):>18} {back:>12.6g} {value - back:>+10.6f}")
    print()


EXAMPLE_VALUES = [1.7, -0.9, 0.01, 5.2]
show_round_trip(EXAMPLE_VALUES, 8, 3)
show_round_trip(EXAMPLE_VALUES, 16, 6)

Look at the table:
- as long as the value is in range, the error is never larger than half a step, and 16 bits give steps 32 times finer than 8 bits
- 0.01 is smaller than half a step of `ap_fixed<8,3>`, and becomes 0: values below the resolution are lost
- 5.2 is out of the range of `ap_fixed<8,3>` and saturates to 3.96875, while it fits in `ap_fixed<16,6>`

Try other values, or other types, if you like. You will meet this conversion again in Part 6: the custom kernel returns the logits as plain integers, which the CPU turns back into floats by dividing them by 2<sup>F</sup>.

### The Precision of the Pair Classifier
The FPGA does not compute with floats but with **fixed-point** numbers, `ap_fixed<W, I>`: *W* bits in total, *I* of them for the integer part (sign included). It holds values from −2<sup>I−1</sup> to 2<sup>I−1</sup>, in steps of 2<sup>−(W−I)</sup>. For every tensor we choose:
- the **integer bits**, from the largest value the tensor takes on real data, so that nothing overflows
- the **total bits**, a trade-off: more bits are more accurate, fewer bits use fewer FPGA resources

### Task 3.1: The Ranges of the Values

In [ ]:
# TODO: measure the largest absolute value of every tensor of the pair classifier

# Hint: run the model on the calibration pairs inside torch.no_grad(): x = torch.from_numpy(calibration_features);
#       model.fc1(x) gives the hidden values (before the ReLU), model(x) the logits; .numpy() converts them
# Hint: np.abs(array).max() is the largest absolute value of an array; the weights are in `weights`

with torch.no_grad():
    x = # YOUR CODE HERE
    hidden_values = # YOUR CODE HERE
    calibration_logits = # YOUR CODE HERE

largest = {"input": None,          # Replace with correct code
           "hidden": None,         # Replace with correct code
           "logits": None,         # Replace with correct code
           "fc1 weight": None,     # Replace with correct code
           "fc1 bias": None,       # Replace with correct code
           "fc2 weight": None,     # Replace with correct code
           "fc2 bias": None}       # Replace with correct code

for name, value in largest.items():
    print(f"{name:11s} largest |value| {value:7.3f}")

### Task 3.2: The Integer Bits
An `ap_fixed<W, I>` holds values up to 2<sup>I−1</sup> (one of the integer bits is the sign): to hold values up to *v* we need the smallest *I* with 2<sup>I−1</sup> > *v*. Every integer bit doubles the range: *I* = 1 holds the values below 1, *I* = 2 below 2, *I* = 3 below 4, and so on. For the activations we add a 25% margin, in case some event goes a bit further than the calibration ones.

In [ ]:
# TODO: write the function that gives the integer bits (sign included) needed for values up to largest_value

# Hint: start from 1 bit (the sign bit), and add one bit at a time while 2 ** (bits - 1) <= largest_value
# Hint: a while loop does it in a few lines

def integer_bits_for(largest_value):
    # YOUR CODE HERE
    return None  # Replace with correct code

ACTIVATION_MARGIN = 1.25                                          # DO NOT CHANGE
integer_bits = {}
for name, value in largest.items():
    is_activation = name in ("input", "hidden", "logits")
    integer_bits[name] = integer_bits_for(value * ACTIVATION_MARGIN if is_activation else value)
    print(f"{name:11s} largest |value| {value:7.3f}  ->  {integer_bits[name]} integer bits")

The cell below turns integer bits and total bits into _hls4ml_ types. We use two total widths, one for the **data** (input, hidden values, logits) and one for the **weights** and biases. Every type rounds to the nearest value (`AP_RND_CONV`) and saturates instead of wrapping around (`AP_SAT`).

In [ ]:
# The types of the pair classifier for given weight bits and data bits (no changes needed)
def fixed_type(total_bits, integer_bits):
    return f"ap_fixed<{total_bits},{integer_bits},AP_RND_CONV,AP_SAT>"


def precision(weight_bits, data_bits):
    return {"input": fixed_type(data_bits, integer_bits["input"]),
            "hidden": fixed_type(data_bits, integer_bits["hidden"]),
            "logits": fixed_type(data_bits, integer_bits["logits"]),
            "w1": fixed_type(weight_bits, integer_bits["fc1 weight"]),
            "b1": fixed_type(weight_bits, integer_bits["fc1 bias"]),
            "w2": fixed_type(weight_bits, integer_bits["fc2 weight"]),
            "b2": fixed_type(weight_bits, integer_bits["fc2 bias"])}


print_dict(precision(16, 18))

## Part 4: Convert, Compile and Simulate
In Exercise 0 you converted the pair classifier with the default settings of _hls4ml_. Now you choose its precision, following the usual workflow of _hls4ml_:
1. one precision for the whole model (Task 4.1)
2. the **profiling**: compare the values of every layer with the range of its type (Task 4.2)
3. one precision for every layer (Task 4.3), with the fewest bits that keep the model accurate (Task 4.4)

Every configuration is checked in **C simulation**: `compile()` builds a C++ emulation of the firmware, and `predict()` runs it, with exactly the fixed-point arithmetic of the FPGA. No synthesis is needed: each check takes about 20 seconds.

### Task 4.1: One Precision for the Whole Model
With `granularity='model'` the configuration has a single precision, `hls_config['Model']['Precision']['default']`, used for every tensor of the model: the input, the weights, the biases, the sums and the results. Try `ap_fixed<8,3>`, the type of the example of Part 3: 8 bits, 3 of them integer bits (sign included).

In [ ]:
# TODO: convert the pair classifier with one precision for the whole model, ap_fixed<8,3>, and run its C simulation

# Hint: hls4ml.utils.config_from_pytorch_model(model, [(64,)], granularity='model', backend='Vivado')
# Hint: uniform_config['Model']['Precision']['default'] = ... sets the precision of the whole model
# Hint: hls4ml.converters.convert_from_pytorch_model(model, hls_config=..., output_dir=..., backend='Vivado',
#                                                     io_type='io_parallel', part=FPGA_PART,
#                                                     clock_period=CLOCK_PERIOD_NS)
#       with output_dir=str(HLS_DIR / 'uniform_8_3')
# Hint: uniform_model.compile() builds the C simulation, uniform_model.predict(test_features) runs it;
#       predict returns a flat array: reshape it to [K, 2] with .reshape(-1, 2)

uniform_config = None     # Replace with correct code
# YOUR CODE HERE: set the precision of the whole model

uniform_model = None      # Replace with correct code
# YOUR CODE HERE: compile it

uniform_logits = None     # Replace with correct code: the C simulation of the test pairs, [K, 2]

print("=" * 20, "HLS Config", "=" * 20)
print_dict(uniform_config["Model"])
print("=" * 50)
show_hls_layers(uniform_model)

The cell below compares the fixed-point model with the float one, pair by pair, on the 171,152 test pairs. For every pair it takes p₁, the probability that the pair is an edge, in the two models, and their difference Δp = p₁(fixed point) − p₁(float). It reports:
- `max_abs_dp`: the largest |Δp|, the pair that moves the most
- `mean_abs_dp`: the mean |Δp|, a typical pair
- `decision_changes`: the pairs that end up on the other side of the threshold (p₁ > 0.1 in one model and not in the other)

A configuration is **accepted** if `max_abs_dp` ≤ `MAX_DP` = 2 × 10⁻³: no probability moves by more than 0.002. Why a limit on the probabilities, and not on the decisions? Because it is what protects the decisions. A decision changes only if p₁ crosses the threshold, so with |Δp| ≤ 0.002 only the pairs with a float p₁ between 0.098 and 0.102 can change: 1,551 of the 171,152 test pairs (0.9%), on which the float model itself is undecided. All the other pairs keep their decision, whatever the fixed-point arithmetic does. The number of decisions that change, instead, depends on how many pairs of this test set happen to be close to the threshold: on its own, it says little about other events.

The plots show the distribution of Δp, and p₁ of the fixed-point model against p₁ of the float one.

In [ ]:
# Compare the fixed-point model with the float one (no changes needed)
MAX_DP = 2e-3          # accepted only if no p1 moves by more than this: the limit on max_abs_dp - DO NOT CHANGE


def probability_of_edge(logits):
    return softmax(logits)[:, 1]


def compare_with_float(fixed_logits):
    p_fixed = probability_of_edge(fixed_logits)
    p_float = probability_of_edge(float_logits)
    differences = np.abs(p_fixed - p_float)
    return {"max_abs_dp": float(differences.max()), "mean_abs_dp": float(differences.mean()),
            "decision_changes": int(((p_fixed > THRESHOLD) != (p_float > THRESHOLD)).sum()),
            "accepted": bool(differences.max() <= MAX_DP)}


print(f"ap_fixed<8,3> everywhere, C simulation of {len(test_features):,} test pairs vs float:")
print_dict(compare_with_float(uniform_logits))
plot_fixed_point_error(probability_of_edge(uniform_logits), probability_of_edge(float_logits), "ap_fixed<8,3>",
                       save_to=FIGURE_DIR / "uniform_8_3_error.png")
print("Plot saved as 'work/figures/uniform_8_3_error.png'")

**Questions**
1. How far does the fixed-point model go from the float one? How many decisions change?
2. Is this what you expected? Compare the range of `ap_fixed<8,3>` (Part 3) with the largest values you measured in Task 3.1: which tensors do not fit?

The cell below tries again with `ap_fixed<16,6>`, the default of _hls4ml_. It puts the steps of Task 4.1 in two functions, `convert_and_compile` and `simulate`, that the next tasks use too.

In [ ]:
# The steps of Task 4.1 in two functions, and one precision for the whole model again: ap_fixed<16,6> (no changes needed)
def convert_and_compile(hls_config, name):
    """Convert the pair classifier with hls_config into HLS_DIR / name, and build its C simulation."""
    hls_model = hls4ml.converters.convert_from_pytorch_model(
        model, hls_config=hls_config, output_dir=str(HLS_DIR / name), backend="Vivado", io_type="io_parallel",
        part=FPGA_PART, clock_period=CLOCK_PERIOD_NS)
    hls_model.compile()
    return hls_model


def simulate(hls_model):
    """The C simulation of the test pairs: the logits [K, 2]."""
    return np.asarray(hls_model.predict(test_features), np.float32).reshape(-1, 2)


default_config = hls4ml.utils.config_from_pytorch_model(model, [(64,)], granularity="model", backend="Vivado")
default_config["Model"]["Precision"]["default"] = "ap_fixed<16,6>"
default_model = convert_and_compile(default_config, "uniform_16_6")
show_hls_layers(default_model)
print("\nap_fixed<16,6> everywhere vs float:")
print_dict(compare_with_float(simulate(default_model)))

`ap_fixed<16,6>` holds values up to 32: nothing overflows any more, and the error drops by a factor 30. But p₁ still moves by up to 3 × 10⁻², and about 3,000 decisions change: one type for everything is a bad compromise. Look at the types above:
- the weights and the biases, all below 0.33, get 6 integer bits: 5 of them are always 0, and could have been fractional bits
- the sums (`accum`) have the same 10 fractional bits as everything else: every product is truncated (`TRN`) before it is added, and the errors add up over the 64 terms of each sum

### Task 4.2: Profiling
With `granularity='name'` the configuration has one entry per layer, in `hls_config['LayerName']`: `x` (the input), `fc1`, `relu` and `fc2`, each with its own `Precision`. The default is `auto`: _hls4ml_ chooses the type itself where it can (for example, the sums get enough bits to be exact), and uses the default precision, `ap_fixed<16,6>`, for the rest. The cell below creates this configuration and turns on the `Trace` of every layer, which the profiling needs: the C simulation then also returns the output of every layer.

In [ ]:
# One configuration entry per layer, with the trace on, and its C simulation (no changes needed)
profiling_config = hls4ml.utils.config_from_pytorch_model(model, [(64,)], granularity="name", backend="Vivado")
for layer in profiling_config["LayerName"].values():
    layer["Trace"] = True                     # the C simulation also returns the output of this layer
print_dict(profiling_config["LayerName"])

profiling_model = convert_and_compile(profiling_config, "profiling_auto")
print()
show_hls_layers(profiling_model)
print("\nauto types vs float:")
print_dict(compare_with_float(simulate(profiling_model)))

With exact sums (`fc1` and `fc2` accumulate in 39 bits) the error is already ten times smaller than with `ap_fixed<16,6>` everywhere. But are the types well chosen? _hls4ml_ has a **profiling** tool to check it, `numerical()` (in `hls4ml.model.profiling`). For every layer it draws the distribution of the absolute values of the weights and biases, and of the outputs on some input data, as box plots on a log₂ scale: the box goes from the first to the third quartile, the whiskers from the smallest to the largest value. The **grey boxes** are the values that the type can represent, from one step, 2<sup>−F</sup>, to the largest value, 2<sup>I−1</sup>:
- **on the right**, the grey box must cover the whole whisker, otherwise the largest values saturate or wrap around
- **on the left**, it can stop before the end of the whisker: the values smaller than one step are lost, but they change the result very little; how far you can go is a matter of trial and error
- a grey box that goes much further right than the whisker wastes integer bits, which could have been fractional bits

In [ ]:
# Profile the weights and the outputs of every layer, on the calibration pairs (no changes needed)
weight_plot, final_weight_plot, _, activation_plot = numerical(model=model, hls_model=profiling_model,
                                                               X=calibration_features)
plt.close(weight_plot)      # the weights before the optimizations of hls4ml: for this model, the same as after
final_weight_plot.savefig(FIGURE_DIR / "profiling_auto_weights.png", bbox_inches="tight")
activation_plot.savefig(FIGURE_DIR / "profiling_auto_outputs.png", bbox_inches="tight")
print("Plots saved as 'work/figures/profiling_auto_weights.png' and 'work/figures/profiling_auto_outputs.png'")

Look at the plots:
- **weights**: they are all below 2⁻¹ (0.33 at most, Task 3.1), but their type, `ap_fixed<16,6>`, reaches 2⁵: the grey boxes run off the right edge of the plot, far beyond the largest weights, while on the left they stop at 2⁻¹⁰, and leave out the smallest weights
- **outputs**: the outputs of `fc1` and `fc2` have the very wide types of the exact sums, far more than their values need; the output of the ReLU has the default type

### Task 4.3: One Precision for Every Layer
In Part 3 you found the integer bits that every tensor needs (`integer_bits`), and `precision(weight_bits, data_bits)` turns them into types for given total widths. Put them in the configuration: complete the function below, which sets the type of every layer of a configuration with `granularity='name'`. Start with generous widths, `precision(16, 18)`: 16 bits for the weights and the biases, 18 for the data (the input, the hidden values and the logits).

In [ ]:
# TODO: complete the function that sets the precision of every layer of a configuration with granularity='name'

# Hint: layers['x']['Precision']['result'] is the type of the input; layers['relu']['Precision']['result'] the one
#       of the hidden values after the ReLU
# Hint: fc1 and fc2 have 'weight', 'bias', 'result' and 'accum' (the sums): the result of fc1 is the hidden values,
#       the result of fc2 the logits; set 'accum' to 'auto', so that hls4ml makes the sums wide enough to be exact
# Hint: the types are in the dictionary `types`: types['input'], types['hidden'], types['logits'], types['w1'], ...

def set_layer_precision(hls_config, types):
    layers = hls_config["LayerName"]
    # YOUR CODE HERE


layer_config = hls4ml.utils.config_from_pytorch_model(model, [(64,)], granularity="name", backend="Vivado")
for layer in layer_config["LayerName"].values():
    layer["Trace"] = True                     # for the profiling and the tracing
set_layer_precision(layer_config, precision(16, 18))
print_dict(layer_config["LayerName"])

In [ ]:
# Convert, simulate and profile the model with one precision for every layer (no changes needed)
layer_model = convert_and_compile(layer_config, "layer_w16_d18")
show_hls_layers(layer_model)
print("\nw16_d18 vs float:")
print_dict(compare_with_float(simulate(layer_model)))

weight_plot, final_weight_plot, _, activation_plot = numerical(model=model, hls_model=layer_model,
                                                               X=calibration_features)
plt.close(weight_plot)
final_weight_plot.savefig(FIGURE_DIR / "profiling_layer_weights.png", bbox_inches="tight")
activation_plot.savefig(FIGURE_DIR / "profiling_layer_outputs.png", bbox_inches="tight")
print("Plots saved as 'work/figures/profiling_layer_weights.png' and 'work/figures/profiling_layer_outputs.png'")

Now every grey box ends at most one power of 2 to the right of its whisker: each type has the integer bits its values need (with the 25% margin for the data), and all its other bits are fractional bits, so the steps are much finer (2⁻¹⁵ for the weights instead of 2⁻¹⁰). The types also round to the nearest value (`RND_CONV`) instead of truncating, and saturate (`SAT`) instead of wrapping around. The largest move of p₁ drops below 10⁻⁴, and only one decision changes.

**Questions**
1. Where do the grey boxes stop on the left now? Compare them with the profiling of Task 4.2.
2. 16 + 18 bits are a lot. Which tensors do you think could lose some bits first?

### Bonus: Tracing
The profiling compares ranges; the **trace** compares values. `hls_model.trace(X)` runs the C simulation and returns, besides the output, the output of every layer that has `Trace` on. Comparing them with the same layers in PyTorch shows in which layer a precision that is too small starts to lose accuracy. Try it also with fewer bits: for example `set_layer_precision(layer_config, precision(8, 10))`, then `layer_model = convert_and_compile(layer_config, "layer_w8_d10")`, and run the cell again.

In [ ]:
# Compare the output of every layer with PyTorch, on 1,000 calibration pairs (no changes needed)
x = calibration_features[:1000]
_, hls_trace = layer_model.trace(x)
with torch.no_grad():
    hidden = model.fc1(torch.from_numpy(x))
    torch_trace = {"fc1": hidden.numpy(), "relu": model.relu(hidden).numpy(), "fc2": model.fc2(model.relu(hidden)).numpy()}

for layer_name, torch_values in torch_trace.items():
    difference = np.abs(hls_trace[layer_name].reshape(torch_values.shape) - torch_values)
    print(f"{layer_name:5s} |hls4ml - PyTorch|: largest {difference.max():.1e}, mean {difference.mean():.1e}")
print(f"\nfc1, first pair, first five values:\n  PyTorch {torch_trace['fc1'][0, :5]}\n"
      f"  hls4ml  {hls_trace['fc1'].reshape(len(x), -1)[0, :5]}")

### Task 4.4: How Few Bits?
Every bit costs FPGA resources: the fewer the bits, the smaller the multipliers and the adders. The cell below converts, compiles and simulates six configurations of (weight bits, data bits), from the fewest total bits to the most (about two minutes). `make_hls_model` does what you did in Task 4.3, and also sets the strategy and the reuse factor that you will use in Part 5.

In [ ]:
# Precision scan: convert, compile and simulate six configurations (no changes needed)
def make_hls_model(types, output_dir, strategy="Latency", reuse_factor=1):
    hls_config = hls4ml.utils.config_from_pytorch_model(model, [(64,)], granularity="name", backend="Vivado",
                                                        default_reuse_factor=reuse_factor)
    hls_config["Model"]["Strategy"] = strategy
    set_layer_precision(hls_config, types)
    for layer_name in ("fc1", "fc2"):
        hls_config["LayerName"][layer_name]["ReuseFactor"] = reuse_factor
        hls_config["LayerName"][layer_name]["Strategy"] = strategy
    return hls4ml.converters.convert_from_pytorch_model(
        model, hls_config=hls_config, output_dir=str(output_dir), backend="Vivado", io_type="io_parallel",
        part=FPGA_PART, clock_period=CLOCK_PERIOD_NS, project_name="myproject")


# (weight bits, data bits), from the fewest total bits to the most (between equal totals, fewer data bits first)
CONFIGURATIONS = [(12, 12), (12, 14), (8, 18), (12, 16), (10, 18), (14, 18)]
scan_results = {}
for weight_bits, data_bits in CONFIGURATIONS:
    name = f"w{weight_bits}_d{data_bits}"
    scan_model = make_hls_model(precision(weight_bits, data_bits), HLS_DIR / f"scan_{name}")
    scan_model.compile()
    result = compare_with_float(simulate(scan_model))
    scan_results[(weight_bits, data_bits)] = result
    print(f"{name:8s} {weight_bits + data_bits} bits  max|Δp| {result['max_abs_dp']:.1e}  "
          f"mean {result['mean_abs_dp']:.1e}  decision changes {result['decision_changes']:3d}  "
          f"{'ACCEPTED' if result['accepted'] else '-'}")

In [ ]:
# TODO: choose the smallest accepted configuration

# Hint: CONFIGURATIONS goes from the fewest bits to the most, so the smallest accepted one is the first accepted one
# Hint: scan_results[(weight_bits, data_bits)]['accepted'] says if a configuration is accepted

WEIGHT_BITS, DATA_BITS = None, None    # Replace with correct code

TYPES = precision(WEIGHT_BITS, DATA_BITS)
print(f"Selected: w{WEIGHT_BITS}_d{DATA_BITS}")
plot_precision_scan(scan_results, (WEIGHT_BITS, DATA_BITS), MAX_DP, save_to=FIGURE_DIR / "precision_scan.png")
print("Plot saved as 'work/figures/precision_scan.png'")

The smallest accepted configuration is w12_d14: 12 bits for the weights and the biases, 14 for the data. Compare it with `ap_fixed<16,6>` everywhere (Task 4.1): fewer bits, and p₁ moves 20 times less. Choosing the precision layer by layer, from the ranges of the values, is what makes the difference.

### Task 4.5: The Final Model
The cell below converts the pair classifier at the selected precision and looks at what _hls4ml_ produced: the types of every layer, and the C++ of the firmware (`firmware/myproject.cpp` calls one function per layer).

In [ ]:
# Convert the final model and inspect it (no changes needed)
FINAL_PROJECT = HLS_DIR / "pair_classifier"
shutil.rmtree(FINAL_PROJECT, ignore_errors=True)
hls_model = make_hls_model(TYPES, FINAL_PROJECT)
hls_model.compile()
show_hls_layers(hls_model)

firmware_dir = FINAL_PROJECT / "firmware"
print(f"\nfirmware/: {sorted(path.name for path in firmware_dir.iterdir())}")
print((firmware_dir / "myproject.cpp").read_text().split("// hls-fpga-machine-learning insert layers")[1].strip()[:700])

hls_logits = simulate(hls_model)
print(f"\nC simulation vs float:")
print_dict(compare_with_float(hls_logits))

In [ ]:
# Visualize the fixed-point error and the types (no changes needed)
plot_fixed_point_error(probability_of_edge(hls_logits), probability_of_edge(float_logits),
                       f"w{WEIGHT_BITS}_d{DATA_BITS}", save_to=FIGURE_DIR / "fixed_point_error.png")
plot_bit_layout(TYPES, save_to=FIGURE_DIR / "bit_layout.png")
print("Plots saved as 'work/figures/fixed_point_error.png' and 'work/figures/bit_layout.png'")

## Part 5: Resources and Latency
The **C-synthesis** of Vitis HLS turns the C++ into a hardware design, and reports, for the U55C at 300 MHz:
- the **latency**: the clocks from an input to its output
- the **interval**: the clocks between two new inputs (1: a new pair every clock)
- the **resources**: DSP slices (the multipliers), LUTs, flip-flops (FF), block RAMs

Two settings of _hls4ml_ decide how parallel the design is:
- `Strategy = "Latency"`: every multiplication gets its own multiplier
- `Strategy = "Resource"` with a **reuse factor** R: each multiplier is shared by R multiplications, so R times fewer multipliers, but a new input only every R clocks

**Important:** each synthesis takes a few minutes. Run the two tasks now!

### Task 5.1: Synthesize the Latency Version

In [ ]:
# TODO: run the C-synthesis of the pair classifier with the Latency strategy and reuse factor 1
import os
os.environ["PATH"] = "/opt/tools/Xilinx/Vitis_HLS/2023.2/bin:/opt/xilinx/xrt/bin:" + os.environ["PATH"]

# Hint: make_hls_model(TYPES, output_dir, strategy='Latency', reuse_factor=1), with output_dir=SYNTH_DIR
# Hint: hls_model.write() writes the project, hls_model.build(csim=False, synth=True, export=False) runs Vitis HLS

SYNTH_DIR = HLS_DIR / "csynth_latency_rf1"
shutil.rmtree(SYNTH_DIR, ignore_errors=True)                # start from an empty folder
# YOUR CODE HERE

synthesis_results = {}
synthesis_results["Latency, RF 1"] = read_csynth_report(SYNTH_DIR / "myproject_prj" / "solution1" / "syn" / "report"
                                                        / "myproject_csynth.xml")
print_dict(synthesis_results["Latency, RF 1"])

### Task 5.2: Synthesize the Resource Version

In [ ]:
# TODO: run the C-synthesis of the pair classifier with the Resource strategy and reuse factor 16

# Hint: the same as Task 5.1, with strategy='Resource', reuse_factor=16 and output_dir=SYNTH_DIR

SYNTH_DIR = HLS_DIR / "csynth_resource_rf16"
shutil.rmtree(SYNTH_DIR, ignore_errors=True)
# YOUR CODE HERE

synthesis_results["Resource, RF 16"] = read_csynth_report(SYNTH_DIR / "myproject_prj" / "solution1" / "syn"
                                                          / "report" / "myproject_csynth.xml")
print_dict(synthesis_results["Resource, RF 16"])

In [ ]:
# Compare the two versions (no changes needed)
plot_synthesis_variants(synthesis_results, save_to=FIGURE_DIR / "synthesis_variants.png")
print("Plot saved as 'work/figures/synthesis_variants.png'")

The Latency version takes a new pair every clock, but every one of the 4,224 multiplications of a pair gets its own hardware: 3,117 DSP slices, and the others are built from LUTs (the weights are constants, and a multiplication by a constant can often be done with a few shifts and additions). The Resource version with reuse factor 16 uses about 12 times fewer DSP slices, but takes a new pair only every 16 clocks. This is the main trade-off of FPGA design: **resources against speed**.

## Part 6: Connecting the Custom Kernel
This part has no code to run: just read it. The kernel that goes on the FPGA is already written, and the bitstream of Part 7 is built from it. Here we see why it is needed, how it works, and how it uses the _hls4ml_ network that you built in Parts 4 and 5.

### Why a Custom Kernel
At one pair per clock, the pair classifier could score a median event in about 4 µs, but only if the pairs reach it that fast. If the pairs are built on the CPU, every pair is 64 floats: for a median event (51 hits, 1,326 pairs), about 340 kB to send to the card. The data the pairs are made from is much smaller: the 51 embeddings of 32 floats, 6.5 kB. The number of pairs grows as N<sup>2</sup>, the number of embeddings only as N:

![Data sent to the FPGA for every event, as a function of the number of hits](../images/data_per_event.png)

So the FPGA receives the embeddings and builds the pairs itself. This is not a job for _hls4ml_, which converts a network with an input of fixed size: the number of hits changes from event to event, and building the pairs is not a layer. The pairs are built by a **custom C++ kernel**, `custom_kernel/pairedges_fact_kernel.cpp`, with the _hls4ml_ network inside. **This kernel is already written for you**: you do not need to change it.

### How the Kernel Works
The first layer of the pair classifier, fc1, acts on the two embeddings put side by side, [e<sub>i</sub>, e<sub>j</sub>]. Before the ReLU it is linear, so it splits into a part that depends only on hit i and a part that depends only on hit j:

z<sub>ij</sub> = W<sub>1</sub> [e<sub>i</sub>, e<sub>j</sub>] + b<sub>1</sub> = (A e<sub>i</sub> + b<sub>1</sub>) + B e<sub>j</sub> = U[i] + V[j]

where A and B are the two halves of W<sub>1</sub>: the 32 columns that multiply e<sub>i</sub>, and the 32 that multiply e<sub>j</sub>. U and V are computed **once per hit** (51 times for a median event) instead of once per pair (1,326 times), and a pair then costs only 64 additions, the ReLU and fc2. The kernel is a chain of loops:

![The stages of the custom kernel](../images/kernel_pipeline.png)

- **LoadHits** reads the embeddings of the event from the card memory, 16 floats per 512-bit word, and converts them to the input type of the network
- **Hits** computes U and V for every hit, 2 of its 32 inputs per clock: 256 multipliers, 16 steps per hit
- **PairLoop** takes one pair (i, j) per clock: U[i] + V[j], the ReLU and fc2, with 128 multipliers
- the **candidate filter**, at the end of PairLoop, keeps only the pairs that can pass the threshold: the **candidates** (see below)
- **WriteCand** writes the candidates to the card memory, 8 per 512-bit word

The kernel does not call the layer functions of _hls4ml_, because it needs its own loops to split fc1. But inside the loops it uses exactly what _hls4ml_ generated for your model (the green boxes): the fixed-point types, the quantized weights, the multiplications and the casts with their roundings. The products and the sums are integer operations, exact whatever their order (Part 3), so the kernel computes the **same logits as your _hls4ml_ model, bit for bit**.

**The candidates.** The card does not compute the softmax: exponentials are expensive in hardware. With two classes, p<sub>1</sub> = e<sup>l<sub>1</sub></sup> / (e<sup>l<sub>0</sub></sup> + e<sup>l<sub>1</sub></sup>) = 1 / (1 + e<sup>−(l<sub>1</sub> − l<sub>0</sub>)</sup>), so p<sub>1</sub> > 0.1 exactly when l<sub>1</sub> − l<sub>0</sub> > ln(0.1 / 0.9) ≈ −2.197: a subtraction and a comparison, done on the integers of the fixed-point logits. The threshold is lowered by a tiny margin (10<sup>−3</sup>), so that no edge is lost to a rounding: the candidates are all the edges and a few more pairs, and the CPU computes p<sub>1</sub> on them and applies the real threshold. Every candidate is a 64-bit word with i, j and the two logits as integers q. The CPU turns them back into floats as in Part 3, l = q × 2<sup>−F</sup>, with F = 10 fractional bits for the logits of w12_d14 (`ap_fixed<14,4>`).

### How the Kernel Finds the _hls4ml_ Code
To use the code of _hls4ml_, the kernel needs the names of its objects: the configurations of the layers, the types, the weight arrays. _hls4ml_ numbers them by the position of the layer in the model, so the names depend on the model and on the frontend (Keras or PyTorch). They can be read in the top function that _hls4ml_ generated, `firmware/myproject.cpp` (you printed it in Task 4.5), where every layer is one call:

```cpp
nnet::dense<input_t, layer2_t, config2>(x, layer2_out, w2, b2); // fc1
nnet::relu<layer2_t, layer3_t, relu_config3>(layer2_out, layer3_out); // relu
nnet::dense<layer3_t, result_t, config4>(layer3_out, layer4_out, w4, b4); // fc2
```

For fc1, for example, `layer2_t` is its output type, `config2` its configuration (with the types of the weights and of the sums, and the multiplication to use), `w2` and `b2` its weights and biases. These names are written into a small header, `pairhead_names.h`, which the kernel includes. For your model it is:

```cpp
// hls4ml names used by the kernel
#pragma once
#define PH_FC1_CONFIG config2
#define PH_FC2_CONFIG config4
#define PH_FC1_OUT_T layer2_t
#define PH_RELU_OUT_T layer3_t
#define PH_W1 w2
#define PH_B1 b2
#define PH_W2 w4
#define PH_B2 b4
#define PH_W1_FILE "w2.txt"
#define PH_B1_FILE "b2.txt"
#define PH_W2_FILE "w4.txt"
#define PH_B2_FILE "b4.txt"
```

The kernel only uses these macros (`PH_W1`, `PH_FC1_CONFIG`, ...), so the same source works with any precision chosen in Part 4: the types and the weights change, the names do not. The `_FILE` macros are the text files from which the C simulation reads the weights. The `firmware/` folder of _hls4ml_, the header and the kernel source are then compiled together:
- with `g++`, into the **C simulation** of the kernel: a library that Python can call with the embeddings of an event, and that returns the candidates as the FPGA would
- with **Vitis HLS** (`v++ --compile`, about 2 minutes), into the hardware description of the kernel, `pairedges.xo`: the starting point of the bitstream (Part 7)

### The Check: the Kernel against Your _hls4ml_ Model
The C simulation of the kernel was compared with your _hls4ml_ model on 100 test events:
- the kernel received the embeddings of each event, and returned its candidates
- the reference built all the pairs on the CPU, ran them through the C simulation of the _hls4ml_ model (`hls_model.predict`), and applied the same candidate test to its logits

The result:

```
171,152 pairs -> 44,996 candidates (26%)
Kernel == hls4ml model, bit for bit: True
```

The same candidates, with the same logits to the last bit. And only about a quarter of the pairs go back to the CPU, 8 bytes each.

### The C-Synthesis of the Kernel
The C-synthesis of the whole kernel reports, for every loop, how often it takes a new item, and the resources of the kernel:

```
Pipelining result : Target II = 1, Final II = 1, Depth = 7, loop 'LoadHits'
Pipelining result : Target II = 1, Final II = 1, Depth = 7, loop 'Hits_Inputs'
Pipelining result : Target II = 1, Final II = 1, Depth = 13, loop 'PairLoop'
Pipelining result : Target II = 1, Final II = 1, Depth = 4, loop 'WriteCand'
{'DSP': 359, 'LUT': 89302, 'FF': 38728, 'BRAM_18K': 192, 'URAM': 16}
```

- every loop takes a new item every clock: `II` is the *initiation interval*, the interval of Part 5. The items are a word of embeddings, 2 inputs of a hit (`Hits_Inputs` is the loop over the inputs inside the loop over the hits), a pair, a word of candidates
- `Depth` is how many clocks one item takes to go through the loop: with a new item every clock, the items overlap, as on an assembly line
- the kernel uses **359 DSP slices**: the 256 multipliers of the per-hit stage and the 128 of fc2 (a few multiplications by constant weights are built from LUTs, as in Part 5). In Part 5, the _hls4ml_ model needed 3,117 DSP slices to take one pair per clock (Latency), or 264 to take one pair every 16 clocks (Resource, reuse factor 16). Splitting fc1 gives both: **one pair per clock, with about 9 times fewer DSP slices**

In Part 7 you will compare these estimates with the resources the kernel really uses on the chip.

## Part 7: The Prebuilt Bitstream
The last step, `v++ --link`, places the kernel into the U55C platform (the part of the FPGA that handles PCIe, the card memory and the clocks), then runs synthesis, placement and routing in Vivado, and writes the bitstream, an `.xclbin` file. **It takes about 2 hours**, so we built it for you, from exactly this kernel and this _hls4ml_ model (precision w12_d14).

The cell below downloads the bitstream from the storage of the course (49 MB) and checks that it is the right file. `PREBUILT_BITSTREAM` (in `utils.py`) describes it: how it was linked, the steps of the build, and its settings.

In [ ]:
# Get the prebuilt bitstream (no changes needed)
bitstream_file = download_bitstream(FPGA_DIR)
prebuilt = PREBUILT_BITSTREAM
print(f"\nLink configuration:\n{prebuilt['link_config']}")
print(f"Link command: {prebuilt['link_command']}\n")
print("\n".join(prebuilt["build_steps"]))

### Task 7.1: Inspect the Bitstream
`xclbinutil` (from XRT) describes an `.xclbin`: its kernels, the memory they use, and the clock the kernel really reached after routing.

In [ ]:
# TODO: read the description of the bitstream with xclbinutil

# Hint: subprocess.run(['xclbinutil', '--info', '--input', str(path)], capture_output=True, text=True).stdout
# Hint: the bitstream is bitstream_file

xclbin_info = None  # Replace with correct code

kernel_clock = re.search(r"ulp_ucs_aclk_kernel_00[\s\S]{0,300}?Requested Freq:\s*([0-9.]+)\s*MHz"
                         r"[\s\S]{0,100}?Achieved Freq:\s*([0-9.]+)\s*MHz", xclbin_info)
print(f"Kernel clock: requested {kernel_clock.group(1)} MHz, achieved {kernel_clock.group(2)} MHz")
memories = sorted(set(re.findall(r"Memory:\s+(HBM\[\d+\])", xclbin_info)))
print(f"Compute unit pairedges_1: {'pairedges_1' in xclbin_info}")
print(f"Card memory used by the kernel: {memories}")

### Task 7.2: Estimates vs Reality
How good were the Vitis HLS estimates of Part 6? `PREBUILT_BITSTREAM` has both the estimate of the C-synthesis of the kernel (`hls_estimate`) and the resources the kernel really uses after place and route (`routed`).

In [ ]:
# Compare the Vitis HLS estimate with the real kernel (no changes needed)
estimate = prebuilt["hls_estimate"]     # the C-synthesis of the kernel (Part 6)
routed = prebuilt["routed"]             # after place and route
plot_estimate_vs_routed(estimate, routed, save_to=FIGURE_DIR / "estimate_vs_routed.png")
print("Plot saved as 'work/figures/estimate_vs_routed.png'")

The DSP count is almost exact (359 estimated, 360 used), because every multiplier is placed as written. LUTs, flip-flops and memories are overestimated: Vitis HLS does not know how Vivado will optimise the logic, and it counts the worst case. The estimates are useful to compare designs and to check that a design fits, but the real numbers only come from place and route.

Finally, the cell below checks that the precision you selected in Task 4.4 is the one of the bitstream. If it is, the FPGA computes exactly the logits of your _hls4ml_ model in C simulation (Part 6), and its results are the ones you studied in Part 4.

In [ ]:
# Check the precision (no changes needed)
same_precision = TYPES == prebuilt["precision"]
print(f"Your precision w{WEIGHT_BITS}_d{DATA_BITS}, bitstream {prebuilt['precision_name']}: "
      f"{'the same' if same_precision else 'DIFFERENT: go back to Task 4.4'}")

In **Exercise 3** you will load the bitstream on the Alveo U55C, run the hybrid model (the encoder on the CPU, the pair building and the pair classifier on the FPGA), check that it finds the same edges and the same tracks as the float model on the CPU, and measure how fast it is compared with the CPU.